# 03b. Валидация: обучение, экзамен и проверка

Последовательность работы: подключение → создание проекта и навыков → создание пулов → загрузка заданий. HTML, CSS, JavaScript и инструкции находятся в локальных файлах.

Для установки зависимостей выполните в корне репозитория `python -m pip install -e ".[crowd,notebook]"`. OAuth-токен заказчика задайте в локальном `.env` как `YANDEX_TASKS_OAUTH_TOKEN`.

Ноутбук создаёт проект, навыки и пулы, затем загружает задания. Открытие пулов, проверка ответов и назначение навыков исполнителям выполняются отдельно.

Сначала выполните `03a_create_annotation.ipynb`: он сохраняет `annotation_ids.json` с ID созданного проекта и навыка участия.

In [ ]:
import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path
from uuid import uuid4

import requests
from dotenv import load_dotenv

start = Path.cwd().resolve()
for ROOT in (start, *start.parents):
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "butterfly_bench").is_dir():
        break
else:
    raise RuntimeError("Запустите ноутбук из папки проекта или одной из её подпапок.")
load_dotenv(ROOT / ".env")
api = requests.Session()
api.headers['Authorization'] = f"OAuth {os.environ['YANDEX_TASKS_OAUTH_TOKEN']}"
BASE_URL = 'https://tasks.yandex.ru/api/v1'


def create(path, payload, params=None):
    response = api.post(f'{BASE_URL}/{path}', json=payload, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


In [ ]:
annotation_ids = json.loads((ROOT / 'annotation_ids.json').read_text(encoding='utf-8'))
participation_skill_id = annotation_ids['participation_skill_id']

## Проект

Интерфейс проверки рамок и инструкция находятся в `crowd/review/interface/`.

In [ ]:
interface = ROOT / 'crowd/review/interface'

view_spec = {'type': 'classic',
 'assets': {'script_urls': ['$TOLOKA_ASSETS/js/toloka-handlebars-templates.js']},
 'markup': (interface / 'task.html').read_text(encoding='utf-8'),
 'script': (interface / 'task.js').read_text(encoding='utf-8'),
 'styles': (interface / 'task.css').read_text(encoding='utf-8'),
 'settings': {'showSkip': True,
              'showTimer': True,
              'showTitle': True,
              'showFinish': True,
              'showReward': True,
              'showSubmit': True,
              'showMessage': True,
              'showFullscreen': True,
              'showInstructions': True}}

project_payload = {'public_name': 'Проверка рамок вокруг бабочек',
 'public_description': 'Проверьте, верно ли выделены взрослые бабочки красными рамками.',
 'public_instructions': (interface / 'instructions.html').read_text(encoding='utf-8'),
 'private_comment': f'shad_bbox_review_v1; source_project={annotation_ids["project_id"]}',
 'assignments_issuing_type': 'AUTOMATED',
 'assignments_automerge_enabled': False,
 'localization_config': {'default_language': 'RU', 'additional_languages': []},
 'task_spec': {'input_spec': {'image': {'type': 'url', 'hidden': False, 'required': True},
                              'photo_id': {'type': 'integer', 'hidden': True, 'required': True},
                              'boxes_json': {'type': 'string', 'hidden': False, 'required': True},
                              'worker_status': {'type': 'string',
                                                'hidden': False,
                                                'required': True,
                                                'allowed_values': ['FOUND',
                                                                   'NONE',
                                                                   'UNCERTAIN',
                                                                   'LOAD_ERROR']},
                              'source_assignment_id': {'type': 'string',
                                                       'hidden': True,
                                                       'required': True}},
               'output_spec': {'verdict': {'type': 'string',
                                           'hidden': False,
                                           'required': True,
                                           'allowed_values': ['OK', 'BAD', 'LOAD_ERROR']}},
               'view_spec': view_spec}}

In [ ]:
project = create('projects', project_payload)
project_id = project['id']
project_id

## Навыки ревьюеров

Навык экзамена и навык текущего качества создаются отдельно. Общий `control_history_id` связывает историю контрольных ответов экзамена, рабочего ревью и повторного ревью.

In [ ]:
exam_skill = create('skills', {'name': f'BBox — экзамен ревьюеров ({project_id})',
 'private_comment': 'Доля правильных ответов на 10 заданиях ревью; допуск от 80.',
 'hidden': True,
 'public_name': {'EN': 'BBox — экзамен ревьюеров (11719)'}})
exam_skill_id = exam_skill['id']

quality_skill = create('skills', {'name': f'hp проекта {project_id}',
 'private_comment': 'Процент правильных ответов на последних 10 контрольных заданиях экзамена и '
                    'рабочего ревью.',
 'hidden': True,
 'public_name': {'EN': 'hp проекта 11719'}})
quality_skill_id = quality_skill['id']

control_history_id = str(uuid4())

## Обучение ревьюеров

Обучение: две страницы по 5 заданий, 20 минут на страницу, повтор через один день. Задания и пояснения загружаются следующей ячейкой.

In [ ]:
training_payload = {'project_id': project_id,
 'private_name': 'BBox — обучение ревьюеров v1',
 'inherited_instructions': True,
 'may_contain_adult_content': False,
 'assignment_max_duration_seconds': 1200,
 'mix_tasks_in_creation_order': True,
 'shuffle_tasks_in_task_suite': False,
 'training_tasks_in_task_suite_count': 5,
 'task_suites_required_to_pass': 2,
 'retry_training_after_days': 1}

In [ ]:
training = create('trainings', training_payload)
training_id = training['id']
training_id

### Загрузка обучения

10 заданий с правильными ответами и пояснениями. Настройки обучения объединяют их в две страницы по 5.

In [ ]:
training_tasks = json.loads((ROOT / 'crowd/review/tasks/validation_training.json').read_text(encoding='utf-8'))
for task in training_tasks:
    create('tasks', {'pool_id': training_id, **task}, params={'open_pool': 'false'})

print(f'Загружено обучающих заданий: {len(training_tasks)}')

## Экзамен ревьюеров

Параметры: 1 ₽ за страницу, перекрытие 30, лимит 20 минут.

In [ ]:
exam_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — экзамен ревьюеров v1',
 'public_description': 'Экзамен проверки рамок: 10 фото, допуск от 8 верных ответов.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 1.0,
 'assignment_max_duration_seconds': 1200,
 'auto_accept_solutions': True,
 'auto_accept_period_day': 21,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'category': 'profile', 'key': 'languages', 'operator': 'IN', 'value': 'RU'},
                    {'category': 'skill', 'key': participation_skill_id, 'operator': 'EQ'}]},
 'quality_control': {'training_requirement': {'training_passing_skill_value': 80,
                                              'training_pool_id': training_id},
                     'configs': [{'collector_config': {'type': 'GOLDEN_SET',
                                                       'uuid': control_history_id,
                                                       'parameters': {'history_size': 10}},
                                  'rules': [{'conditions': [{'key': 'golden_set_answers_count',
                                                             'operator': 'GTE',
                                                             'value': 10}],
                                             'action': {'type': 'SET_SKILL_FROM_OUTPUT_FIELD',
                                                        'parameters': {'from_field': 'golden_set_correct_answers_rate',
                                                                       'skill_id': exam_skill_id}}},
                                            {'conditions': [{'key': 'golden_set_answers_count',
                                                             'operator': 'GTE',
                                                             'value': 10}],
                                             'action': {'type': 'SET_SKILL_FROM_OUTPUT_FIELD',
                                                        'parameters': {'from_field': 'golden_set_correct_answers_rate',
                                                                       'skill_id': quality_skill_id}}}]}]},
 'defaults': {'default_overlap_for_new_task_suites': 30},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
exam_pool = create('pools', exam_pool_payload)
exam_pool_id = exam_pool['id']
exam_pool_id

### Загрузка заданий

10 контрольных заданий на одной странице: 5 OK и 5 BAD, перекрытие 30. Правильные ответы передаются в `known_solutions`; пояснения в экзамен не передаются.

In [ ]:
pages = json.loads((ROOT / 'crowd/review/tasks/validation_exam.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': exam_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

Допуск после обучения — от 80. По 10 контрольным ответам платформа записывает процент правильных ответов в оба навыка: экзамен и текущее качество.

## Рабочая валидация 400 фотографий

Параметры: 1 ₽ за страницу, перекрытие 3, лимит 20 минут.

In [ ]:
work_pool_payload = {'project_id': project_id,
 'private_name': 'BBox — рабочее ревью 400 фото',
 'public_description': 'Проверьте рамки вокруг взрослых бабочек. До 10 фотографий на странице.',
 'may_contain_adult_content': False,
 'reward_per_assignment': 1.0,
 'assignment_max_duration_seconds': 1200,
 'auto_accept_solutions': True,
 'auto_accept_period_day': 21,
 'assignments_issuing_config': {'issue_task_suites_in_creation_order': False},
 'filter': {'and': [{'category': 'computed',
                     'key': 'client_type',
                     'operator': 'EQ',
                     'value': 'BROWSER'},
                    {'category': 'profile', 'key': 'languages', 'operator': 'IN', 'value': 'RU'},
                    {'category': 'skill', 'key': participation_skill_id, 'operator': 'EQ'},
                    {'category': 'skill', 'key': exam_skill_id, 'operator': 'GTE', 'value': 80},
                    {'category': 'skill', 'key': quality_skill_id, 'operator': 'GTE', 'value': 80}]},
 'quality_control': {'configs': [{'collector_config': {'type': 'GOLDEN_SET',
                                                       'uuid': control_history_id,
                                                       'parameters': {'history_size': 10}},
                                  'rules': [{'conditions': [{'key': 'golden_set_answers_count',
                                                             'operator': 'GTE',
                                                             'value': 10}],
                                             'action': {'type': 'SET_SKILL_FROM_OUTPUT_FIELD',
                                                        'parameters': {'from_field': 'golden_set_correct_answers_rate',
                                                                       'skill_id': quality_skill_id}}}]}]},
 'defaults': {'default_overlap_for_new_task_suites': 3},
 'will_expire': (datetime.now(timezone.utc) + timedelta(days=30)).strftime('%Y-%m-%dT%H:%M:%SZ')}

In [ ]:
work_pool = create('pools', work_pool_payload)
work_pool_id = work_pool['id']
work_pool_id

### Загрузка заданий

400 ответов разметчиков и 45 контрольных заданий, 45 страниц. Полные страницы содержат 9 рабочих заданий и 1 контрольное; последняя — 4 рабочих и 1 контрольное. Входной файл содержит фотографии, статусы, рамки и `source_assignment_id` для связи проверки с ответом разметчика.

In [ ]:
pages = json.loads((ROOT / 'crowd/review/tasks/validation_work.json').read_text(encoding='utf-8'))
for page in pages:
    create('task-suites', {'pool_id': work_pool_id, **page}, params={'open_pool': 'false'})

print(f'Загружено страниц: {len(pages)}; заданий: {sum(len(p["tasks"]) for p in pages)}')

## Созданные ID

In [ ]:
validation_ids = {
    'project_id': project_id,
    'exam_skill_id': exam_skill_id,
    'quality_skill_id': quality_skill_id,
    'training_id': training_id,
    'exam_pool_id': exam_pool_id,
    'work_pool_id': work_pool_id,
    'control_history_id': control_history_id,
}
(ROOT / 'validation_ids.json').write_text(
    json.dumps(validation_ids, ensure_ascii=False, indent=2), encoding='utf-8'
)
validation_ids